In [1]:

import os
import json
import time
import torch
import pandas as pd

# Dataset path: same dataset used in the previous experiments.
INPUT_DIR = (
    "/kaggle/input/datasets/dishita05/"
    "gen-ai-ia3-language-translation-dataset"
)

# Separate folders so existing experiment results remain untouched.
WORK_DIR = "/kaggle/working/ICT4403_MT_Project"
RESULTS_DIR = os.path.join(WORK_DIR, "results")
MODEL_DIR = os.path.join(WORK_DIR, "models")

GEMMA_RESULTS_DIR = os.path.join(RESULTS_DIR, "gemma")
GEMMA_OUTPUT_DIR = os.path.join(MODEL_DIR, "gemma_lora")

for folder in [GEMMA_RESULTS_DIR, GEMMA_OUTPUT_DIR]:
    os.makedirs(folder, exist_ok=True)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory (GB):",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
    )

print("\nDataset files:")
print(os.listdir(INPUT_DIR))

train_df = pd.read_csv(os.path.join(INPUT_DIR, "train.csv"))
val_df = pd.read_csv(os.path.join(INPUT_DIR, "validation.csv"))
test_df = pd.read_csv(os.path.join(INPUT_DIR, "test.csv"))

assert {"en", "es"}.issubset(train_df.columns)
assert {"en", "es"}.issubset(val_df.columns)
assert {"en", "es"}.issubset(test_df.columns)

print("\nTrain shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nExample:")
print("English:", train_df.iloc[0]["en"])
print("Spanish:", train_df.iloc[0]["es"])

print("\nGemma output directory:", GEMMA_OUTPUT_DIR)
print("Gemma results directory:", GEMMA_RESULTS_DIR)


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory (GB): 14.56

Dataset files:
['test.csv', 'train.csv', 'validation.csv']

Train shape: (20000, 2)
Validation shape: (1000, 2)
Test shape: (1000, 2)

Example:
English: Are you sure it's a kill chip and not a tracker?
Spanish: ¿Estás seguro que es un chip asesino y no un rastreador?

Gemma output directory: /kaggle/working/ICT4403_MT_Project/models/gemma_lora
Gemma results directory: /kaggle/working/ICT4403_MT_Project/results/gemma


In [2]:
# STEP 2 — Install Gemma fine-tuning dependencies

%pip install -q -U \
    transformers \
    accelerate \
    peft \
    bitsandbytes \
    datasets \
    sentencepiece \
    sacrebleu \
    huggingface_hub

import transformers
import peft
import bitsandbytes
import accelerate
import huggingface_hub

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("Accelerate:", accelerate.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

# Check whether Hugging Face authentication is available
from huggingface_hub import get_token

hf_token = get_token()

print("\nHugging Face token available:", hf_token is not None)

if hf_token is None:
    print(
        "Gemma may require authentication. "
        "Accept the model terms on Hugging Face and "
        "configure your HF_TOKEN in Kaggle Secrets."
    )

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
Note: you may need to restart the kernel to use updated packages.
Transformers: 5.21.0
PEFT: 0.21.2
BitsAndBytes: 0.50.2
Accelerate: 1.15.0
Hugging Face Hub: 1.33.0

Hugging Face token available: True


In [3]:
from huggingface_hub import model_info

MODEL_NAME = "google/gemma-2b-it"

try:
    info = model_info(MODEL_NAME)
    print("Model accessible:", info.id)
    print("Gemma access check passed!")
except Exception as e:
    print("Access check failed:", type(e).__name__)
    print(str(e)[:500])

Model accessible: google/gemma-2b-it
Gemma access check passed!


In [4]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
import torch

MODEL_NAME = "google/gemma-2b-it"

# Authenticate securely using Kaggle Secrets
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token, add_to_git_credential=False)

print("Authentication successful.")
print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=hf_token,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading Gemma in 4-bit mode...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=hf_token,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

model.config.use_cache = False

print("\nGemma loaded successfully!")
print("Vocabulary size:", len(tokenizer))
print(
    "GPU memory allocated (GB):",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
)

Authentication successful.
Loading tokenizer...


[transformers] We found `hidden_act="gelu"` in this Gemma config. This is a legacy value of the official releases but it is meant to target the tanh approximation. Setting `hidden_act="gelu_pytorch_tanh"` instead.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading Gemma in 4-bit mode...


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]


Gemma loaded successfully!
Vocabulary size: 256000
GPU memory allocated (GB): 1.93


In [5]:
# STEP 8 — Prepare Gemma for QLoRA fine-tuning

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

# Prepare the quantized base model for adapter training
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

# Display trainable parameter count
model.print_trainable_parameters()

trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
total_params = sum(p.numel() for p in model.parameters())

print("\nTotal parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")
print(
    "Trainable percentage:",
    round(100 * trainable_params / total_params, 4),
    "%",
)

print("\nLoRA adapters configured successfully.")

trainable params: 9,805,824 || all params: 2,515,978,240 || trainable%: 0.3897

Total parameters: 1,525,073,920
Trainable parameters: 9,805,824
Trainable percentage: 0.643 %

LoRA adapters configured successfully.


In [6]:
%pip install -q --force-reinstall --no-deps "pyarrow==20.0.0"

Note: you may need to restart the kernel to use updated packages.


In [7]:
import pyarrow as pa
import datasets
import pandas as pd

print("PyArrow:", pa.__version__)
print("Datasets:", datasets.__version__)
print("Pandas:", pd.__version__)

from datasets import Dataset

print("Dataset import successful!")

PyArrow: 20.0.0
Datasets: 5.1.0
Pandas: 2.3.3
Dataset import successful!


In [8]:
# STEP 9 — Prepare Gemma translation dataset

from datasets import Dataset

MAX_LENGTH = 256

def make_prompt(en_text):
    return (
        "Translate the following English text into Spanish.\n"
        f"English: {en_text}\n"
        "Spanish:"
    )

def preprocess_batch(batch):
    input_ids_list = []
    attention_masks_list = []
    labels_list = []

    for en_text, es_text in zip(batch["en"], batch["es"]):
        prompt = make_prompt(str(en_text))
        target = " " + str(es_text) + tokenizer.eos_token

        # Tokenize prompt and target separately
        prompt_ids = tokenizer(
            prompt,
            add_special_tokens=True,
            truncation=True,
            max_length=MAX_LENGTH,
        )["input_ids"]

        target_ids = tokenizer(
            target,
            add_special_tokens=False,
            truncation=True,
            max_length=MAX_LENGTH,
        )["input_ids"]

        # Reserve space for the target
        target_ids = target_ids[:MAX_LENGTH - len(prompt_ids)]

        input_ids = prompt_ids + target_ids
        labels = ([-100] * len(prompt_ids)) + target_ids
        attention_mask = [1] * len(input_ids)

        input_ids_list.append(input_ids)
        attention_masks_list.append(attention_mask)
        labels_list.append(labels)

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_masks_list,
        "labels": labels_list,
    }

train_dataset = Dataset.from_pandas(
    train_df[["en", "es"]].reset_index(drop=True)
)
val_dataset = Dataset.from_pandas(
    val_df[["en", "es"]].reset_index(drop=True)
)

print("Tokenizing training dataset...")
tokenized_train = train_dataset.map(
    preprocess_batch,
    batched=True,
    batch_size=128,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing train",
)

print("Tokenizing validation dataset...")
tokenized_val = val_dataset.map(
    preprocess_batch,
    batched=True,
    batch_size=128,
    remove_columns=val_dataset.column_names,
    desc="Tokenizing validation",
)

print("\nTraining examples:", len(tokenized_train))
print("Validation examples:", len(tokenized_val))

sample = tokenized_train[0]
print("Sample sequence length:", len(sample["input_ids"]))
print("Prompt tokens masked:", sum(x == -100 for x in sample["labels"]))
print("Spanish target tokens:", sum(x != -100 for x in sample["labels"]))

assert len(tokenized_train) == 20000
assert len(tokenized_val) == 1000
assert all(
    any(label != -100 for label in row["labels"])
    for row in tokenized_train.select(range(100))
)

print("\nDataset preparation completed successfully.")

Tokenizing training dataset...


Tokenizing train:   0%|          | 0/20000 [00:00<?, ? examples/s]

Tokenizing validation dataset...


Tokenizing validation:   0%|          | 0/1000 [00:00<?, ? examples/s]


Training examples: 20000
Validation examples: 1000
Sample sequence length: 44
Prompt tokens masked: 29
Spanish target tokens: 15

Dataset preparation completed successfully.


In [10]:
# STEP 10A — Version-compatible Gemma training setup

import os
import inspect
from transformers import TrainingArguments, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=None,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)

GEMMA_CHECKPOINT_DIR = os.path.join(
    GEMMA_OUTPUT_DIR, "checkpoints"
)
GEMMA_LOG_DIR = os.path.join(
    GEMMA_RESULTS_DIR, "training_logs"
)

os.makedirs(GEMMA_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(GEMMA_LOG_DIR, exist_ok=True)

# Start with the core settings
training_args_dict = {
    "output_dir": GEMMA_CHECKPOINT_DIR,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "learning_rate": 2e-4,
    "logging_steps": 50,
    "save_strategy": "epoch",
    "fp16": True,
    "gradient_checkpointing": True,
    "optim": "paged_adamw_8bit",
    "report_to": "none",
    "save_total_limit": 1,
    "seed": 42,
}

# Add optional arguments only if supported by this installation
supported = inspect.signature(
    TrainingArguments.__init__
).parameters

if "eval_strategy" in supported:
    training_args_dict["eval_strategy"] = "epoch"
elif "evaluation_strategy" in supported:
    training_args_dict["evaluation_strategy"] = "epoch"

if "logging_dir" in supported:
    training_args_dict["logging_dir"] = GEMMA_LOG_DIR

if "logging_strategy" in supported:
    training_args_dict["logging_strategy"] = "steps"

if "gradient_checkpointing_kwargs" in supported:
    training_args_dict["gradient_checkpointing_kwargs"] = {
        "use_reentrant": False
    }

if "use_cpu" in supported:
    training_args_dict["use_cpu"] = False

# Filter against the installed version's supported parameters
training_args_dict = {
    k: v for k, v in training_args_dict.items()
    if k in supported
}

training_args = TrainingArguments(**training_args_dict)

print("Training configuration ready.")
print("Transformers:", __import__("transformers").__version__)
print("Epochs:", training_args.num_train_epochs)
print("Batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print(
    "Effective batch size:",
    training_args.per_device_train_batch_size
    * training_args.gradient_accumulation_steps,
)
print("Learning rate:", training_args.learning_rate)
print("FP16:", training_args.fp16)
print("Gradient checkpointing:", training_args.gradient_checkpointing)
print("Evaluation strategy:", getattr(
    training_args, "eval_strategy",
    getattr(training_args, "evaluation_strategy", "not exposed")
))
print("Checkpoint directory:", GEMMA_CHECKPOINT_DIR)

Training configuration ready.
Transformers: 5.21.0
Epochs: 1
Batch size: 2
Gradient accumulation: 8
Effective batch size: 16
Learning rate: 0.0002
FP16: True
Gradient checkpointing: True
Evaluation strategy: IntervalStrategy.EPOCH
Checkpoint directory: /kaggle/working/ICT4403_MT_Project/models/gemma_lora/checkpoints


In [11]:
# STEP 11 — Initialize Gemma Trainer

import inspect
import torch
from transformers import Trainer

# Confirm GPU is available
assert torch.cuda.is_available(), "GPU is not available."
assert len(tokenized_train) == 20000
assert len(tokenized_val) == 1000

# Check that the model has trainable LoRA parameters
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
assert trainable_params > 0, "No trainable parameters found."

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized_train,
    "eval_dataset": tokenized_val,
    "data_collator": data_collator,
}

# Add tokenizer/processing class only if supported
trainer_signature = inspect.signature(Trainer.__init__).parameters

if "processing_class" in trainer_signature:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_signature:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

print("Trainer initialized successfully.")
print("Training examples:", len(trainer.train_dataset))
print("Validation examples:", len(trainer.eval_dataset))
print("Trainable parameters:", f"{trainable_params:,}")
print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory allocated (GB):",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
)
print("\nReady to start Gemma fine-tuning.")

Trainer initialized successfully.
Training examples: 20000
Validation examples: 1000
Trainable parameters: 9,805,824
GPU: Tesla T4
GPU memory allocated (GB): 2.94

Ready to start Gemma fine-tuning.


In [12]:
# STEP 12 — Train Gemma with QLoRA

import time
import json
import torch
from datetime import datetime

# Record starting information
training_start_time = time.time()
training_start_timestamp = datetime.now().isoformat()

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

print("Starting Gemma QLoRA training...")
print("Start time:", training_start_timestamp)
print("Training examples:", len(tokenized_train))
print("Epochs:", training_args.num_train_epochs)
print("Please allow the training to complete.")

train_result = trainer.train()

training_end_time = time.time()
training_end_timestamp = datetime.now().isoformat()
gemma_training_seconds = training_end_time - training_start_time

# Collect measured training statistics
gemma_training_summary = {
    "model": MODEL_NAME,
    "method": "4-bit QLoRA",
    "train_examples": len(tokenized_train),
    "validation_examples": len(tokenized_val),
    "epochs": training_args.num_train_epochs,
    "trainable_parameters": int(trainable_params),
    "training_seconds": round(gemma_training_seconds, 2),
    "training_minutes": round(gemma_training_seconds / 60, 2),
    "start_timestamp": training_start_timestamp,
    "end_timestamp": training_end_timestamp,
    "train_metrics": {
        k: float(v) if isinstance(v, (int, float)) else str(v)
        for k, v in train_result.metrics.items()
    },
}

if torch.cuda.is_available():
    gemma_training_summary["peak_gpu_memory_gb"] = round(
        torch.cuda.max_memory_allocated() / 1024**3, 2
    )

summary_path = os.path.join(
    GEMMA_RESULTS_DIR, "gemma_training_summary.json"
)

with open(summary_path, "w") as f:
    json.dump(gemma_training_summary, f, indent=2)

print("\nTraining completed.")
print("Training duration (minutes):", gemma_training_summary["training_minutes"])
print("Training metrics:", train_result.metrics)
print("Summary saved to:", summary_path)

if "peak_gpu_memory_gb" in gemma_training_summary:
    print(
        "Peak GPU memory (GB):",
        gemma_training_summary["peak_gpu_memory_gb"],
    )

Starting Gemma QLoRA training...
Start time: 2026-10-28T08:53:14.472049
Training examples: 20000
Epochs: 1
Please allow the training to complete.


Epoch,Training Loss,Validation Loss
1,3.045228,3.124091



Training completed.
Training duration (minutes): 51.92
Training metrics: {'train_runtime': 3114.77, 'train_samples_per_second': 6.421, 'train_steps_per_second': 0.201, 'total_flos': 1.911253041291264e+16, 'train_loss': 3.2548715698242185, 'epoch': 1.0}
Summary saved to: /kaggle/working/ICT4403_MT_Project/results/gemma/gemma_training_summary.json
Peak GPU memory (GB): 7.06


In [14]:
# STEP 15 — Generate predictions on the held-out test set

import os
import time
import json
import torch
import pandas as pd
from tqdm.auto import tqdm

model.eval()
model.config.use_cache = True

MAX_NEW_TOKENS = 96
BATCH_SIZE = 4

predictions = []
references = []
inference_start = time.time()

for start in tqdm(range(0, len(test_df), BATCH_SIZE), desc="Gemma inference"):
    batch_df = test_df.iloc[start:start + BATCH_SIZE]

    prompts = [
        make_prompt(str(en))
        for en in batch_df["en"].tolist()
    ]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(model.device)

    prompt_width = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

    # Decoder-only models return prompt tokens + generated tokens.
    generated_only = generated[:, prompt_width:]

    decoded = tokenizer.batch_decode(
        generated_only,
        skip_special_tokens=True,
    )

    predictions.extend([text.strip() for text in decoded])
    references.extend(batch_df["es"].astype(str).tolist())

    del inputs, generated, generated_only

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if torch.cuda.is_available():
    torch.cuda.synchronize()

inference_seconds = time.time() - inference_start

assert len(predictions) == len(test_df) == 1000
assert len(references) == len(test_df)

print("Test examples:", len(predictions))
print("Total inference time (seconds):", round(inference_seconds, 2))
print("Inference time per example (seconds):", round(inference_seconds / len(predictions), 4))

print("\nSample predictions:")
for i in range(min(5, len(predictions))):
    print(f"\nExample {i + 1}")
    print("EN:", test_df.iloc[i]["en"])
    print("Reference ES:", references[i])
    print("Gemma ES:", predictions[i])

# Save predictions and timing
prediction_df = pd.DataFrame({
    "en": test_df["en"].astype(str).tolist(),
    "reference_es": references,
    "gemma_prediction": predictions,
})

prediction_path = os.path.join(
    GEMMA_RESULTS_DIR, "gemma_predictions.csv"
)
prediction_df.to_csv(prediction_path, index=False)

inference_summary = {
    "model": MODEL_NAME,
    "test_ex4amples": len(predictions),
    "decoding": "greedy",
    "do_sample": False,
    "num_beams": 1,
    "max_new_tokens": MAX_NEW_TOKENS,
    "batch_size": BATCH_SIZE,
    "total_inference_seconds": round(inference_seconds, 2),
    "seconds_per_example": round(inference_seconds / len(predictions), 4),
}

with open(
    os.path.join(GEMMA_RESULTS_DIR, "gemma_inference_summary.json"), "w"
) as f:
    json.dump(inference_summary, f, indent=2)

print("\nPredictions saved:", prediction_path)
print("Inference summary saved.")

Gemma inference:   0%|          | 0/250 [00:00<?, ?it/s]

Test examples: 1000
Total inference time (seconds): 656.92
Inference time per example (seconds): 0.6569

Sample predictions:

Example 1
EN: We humans will inevitably end up controlling our own evolution, and, because our power is emergent from nature we will make use of this acquired capacity sooner or later, for better or worse.
Reference ES: Nosotros humanos terminaremos inevitablemente controlando nuestra propia evolución y, puesto que nuestro poder emerge de la naturaleza, tarde o temprano utilizaremos esa capacidad adquirida, para bien o para mal.
Gemma ES: Nosotros, en fin, terminaremos con nosotros mismos controlando nuestra propia evolución, y, porque nuestra capacidad es emergente de la naturaleza, utilizaremos esta capacidad antes o después, para bien o mal.

Example 2
EN: If she had not rented quarter in this building our lives would have be very different.
Reference ES: Si no hubiera rentado un cuarto en este edificio nuestras vidas habrían sido muy diferentes.
Gemma ES: Si

In [13]:
# STEP 14 — Save trained Gemma adapter and tokenizer

import os
import json

FINAL_ADAPTER_DIR = os.path.join(
    GEMMA_OUTPUT_DIR, "final_adapter"
)
os.makedirs(FINAL_ADAPTER_DIR, exist_ok=True)

# Save only the trained adapter, not a duplicate full base model
model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)

# Save a compact experiment configuration
gemma_config = {
    "base_model": MODEL_NAME,
    "method": "QLoRA",
    "quantization": "4-bit NF4 with double quantization",
    "lora_rank": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "max_sequence_length": MAX_LENGTH,
    "epochs": 1,
    "learning_rate": 2e-4,
    "train_examples": len(tokenized_train),
    "validation_examples": len(tokenized_val),
    "trainable_parameters": int(trainable_params),
    "training_minutes": gemma_training_summary["training_minutes"],
    "peak_gpu_memory_gb": gemma_training_summary.get(
        "peak_gpu_memory_gb"
    ),
}

with open(
    os.path.join(GEMMA_RESULTS_DIR, "gemma_config.json"), "w"
) as f:
    json.dump(gemma_config, f, indent=2)

print("Gemma adapter saved:", FINAL_ADAPTER_DIR)
print("Files:", sorted(os.listdir(FINAL_ADAPTER_DIR)))
print("Experiment config saved.")

Gemma adapter saved: /kaggle/working/ICT4403_MT_Project/models/gemma_lora/final_adapter
Files: ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'tokenizer.json', 'tokenizer_config.json']
Experiment config saved.


In [15]:
# STEP 17 — Evaluate Gemma using BLEU and chrF

import os
import json
import sacrebleu
import pandas as pd

# Confirm prediction and reference alignment
assert len(predictions) == 1000
assert len(references) == 1000
assert len(predictions) == len(references)

# Corpus-level metrics; SacreBLEU reports scores on a 0–100 scale
bleu_result = sacrebleu.corpus_bleu(
    predictions,
    [references],
)

chrf_result = sacrebleu.corpus_chrf(
    predictions,
    [references],
)

gemma_metrics = {
    "model": MODEL_NAME,
    "method": "4-bit QLoRA",
    "test_examples": len(predictions),
    "bleu": round(float(bleu_result.score), 4),
    "chrf": round(float(chrf_result.score), 4),
    "inference_seconds": round(float(inference_seconds), 2),
    "seconds_per_example": round(
        float(inference_seconds) / len(predictions), 4
    ),
    "decoding": "greedy",
    "max_new_tokens": MAX_NEW_TOKENS,
    "sacrebleu_version": sacrebleu.__version__,
}

metrics_path = os.path.join(
    GEMMA_RESULTS_DIR, "gemma_metrics.json"
)

with open(metrics_path, "w") as f:
    json.dump(gemma_metrics, f, indent=2)

print("GEMMA EVALUATION RESULTS")
print("=" * 35)
print(f"BLEU: {gemma_metrics['bleu']:.4f}")
print(f"chrF: {gemma_metrics['chrf']:.4f}")
print(f"Test examples: {gemma_metrics['test_examples']}")
print(f"Inference time: {gemma_metrics['inference_seconds']:.2f} seconds")
print(f"Time per example: {gemma_metrics['seconds_per_example']:.4f} seconds")
print("\nSaved metrics:", metrics_path)
print("\nBLEU signature:", bleu_result.format())
print("chrF signature:", chrf_result.format())

GEMMA EVALUATION RESULTS
BLEU: 29.4843
chrF: 53.2853
Test examples: 1000
Inference time: 656.92 seconds
Time per example: 0.6569 seconds

Saved metrics: /kaggle/working/ICT4403_MT_Project/results/gemma/gemma_metrics.json

BLEU signature: BLEU = 29.48 59.6/35.9/24.2/16.7 (BP = 0.967 ratio = 0.967 hyp_len = 14882 ref_len = 15386)
chrF signature: chrF2 = 53.29


In [16]:
# STEP 19 — Back up Gemma adapter and experiment results

import os
import shutil
import json

BACKUP_ROOT = "/kaggle/working/gemma_experiment_backup"
os.makedirs(BACKUP_ROOT, exist_ok=True)

# Include the final adapter, not intermediate checkpoints
adapter_backup_dir = os.path.join(BACKUP_ROOT, "final_adapter")
results_backup_dir = os.path.join(BACKUP_ROOT, "results")

os.makedirs(adapter_backup_dir, exist_ok=True)
os.makedirs(results_backup_dir, exist_ok=True)

# Copy adapter files
for filename in os.listdir(FINAL_ADAPTER_DIR):
    src = os.path.join(FINAL_ADAPTER_DIR, filename)
    dst = os.path.join(adapter_backup_dir, filename)

    if os.path.isfile(src):
        shutil.copy2(src, dst)

# Copy Gemma experiment results
for filename in os.listdir(GEMMA_RESULTS_DIR):
    src = os.path.join(GEMMA_RESULTS_DIR, filename)
    dst = os.path.join(results_backup_dir, filename)

    if os.path.isfile(src):
        shutil.copy2(src, dst)

# Create a ZIP archive
backup_zip = shutil.make_archive(
    "/kaggle/working/gemma_experiment_backup",
    "zip",
    root_dir=BACKUP_ROOT,
)

print("Backup created:", backup_zip)
print("ZIP size (MB):", round(os.path.getsize(backup_zip) / 1024**2, 2))
print("\nBackup contents:")
for root, dirs, files in os.walk(BACKUP_ROOT):
    for filename in files:
        print(os.path.relpath(os.path.join(root, filename), BACKUP_ROOT))

Backup created: /kaggle/working/gemma_experiment_backup.zip
ZIP size (MB): 39.96

Backup contents:
results/gemma_inference_summary.json
results/gemma_metrics.json
results/gemma_config.json
results/gemma_training_summary.json
results/gemma_predictions.csv
final_adapter/adapter_model.safetensors
final_adapter/tokenizer_config.json
final_adapter/adapter_config.json
final_adapter/tokenizer.json
final_adapter/chat_template.jinja
final_adapter/README.md


In [17]:
# STEP 20 — Consolidate all model results

import os
import json
import pandas as pd

WORK_DIR = "/kaggle/working/ICT4403_MT_Project"
RESULTS_DIR = os.path.join(WORK_DIR, "results")

# Values from the completed experiments
comparison = [
    {
        "Model": "Basic Transformer",
        "BLEU": 13.0148,
        "chrF": 6.1098,
        "Training Time (min)": 2.54,
        "Inference Time (sec/1000)": 73.141,
        "Trainable Parameters": 4783232,
        "Decoding": "Greedy",
    },
    {
        "Model": "mT5-small",
        "BLEU": 1.5833,
        "chrF": 11.4274,
        "Training Time (min)": 26.89,
        "Inference Time (sec/1000)": 124.40,
        "Trainable Parameters": 300176768,
        "Decoding": "Greedy",
    },
    {
        "Model": "OPUS-MT (baseline)",
        "BLEU": 42.5432,
        "chrF": 64.8982,
        "Training Time (min)": 0.0,
        "Inference Time (sec/1000)": 43.27,
        "Trainable Parameters": 77943296,
        "Decoding": "Beam search (4 beams)",
    },
    {
        "Model": "TinyLlama 1.1B",
        "BLEU": 23.8893,
        "chrF": 48.2915,
        "Training Time (min)": 62.96,
        "Inference Time (sec/1000)": 1150.44,
        "Trainable Parameters": 2252800,
        "Decoding": "Greedy",
    },
    {
        "Model": "Gemma 2B IT",
        "BLEU": 29.4843,
        "chrF": 53.2853,
        "Training Time (min)": 51.92,
        "Inference Time (sec/1000)": 656.92,
        "Trainable Parameters": 9805824,
        "Decoding": "Greedy",
    },
]

comparison_df = pd.DataFrame(comparison)

comparison_path = os.path.join(
    RESULTS_DIR, "final_model_comparison.csv"
)
comparison_df.to_csv(comparison_path, index=False)

print("Consolidated comparison:")
display(comparison_df)

print("\nSaved to:", comparison_path)

Consolidated comparison:


,Model,BLEU,chrF,Training Time (min),Inference Time (sec/1000),Trainable Parameters,Decoding
0,Basic Transformer,13.0148,6.1098,2.54,73.141,4783232,Greedy
1,mT5-small,1.5833,11.4274,26.89,124.400,300176768,Greedy
2,OPUS-MT (baseline),42.5432,64.8982,0.00,43.270,77943296,Beam search (4 beams)
3,TinyLlama 1.1B,23.8893,48.2915,62.96,1150.440,2252800,Greedy
4,Gemma 2B IT,29.4843,53.2853,51.92,656.920,9805824,Greedy



Saved to: /kaggle/working/ICT4403_MT_Project/results/final_model_comparison.csv


In [18]:
print("Model available:", "model" in globals())
print("Tokenizer available:", "tokenizer" in globals())
print("Test set available:", "test_df" in globals())
print("Predictions available:", "predictions" in globals())

if "model" in globals():
    print("Model device:", model.device)

if "test_df" in globals():
    print("Test examples:", len(test_df))
    print("\nExample English:", test_df.iloc[0]["en"])
    print("Reference Spanish:", test_df.iloc[0]["es"])

Model available: True
Tokenizer available: True
Test set available: True
Predictions available: True
Model device: cuda:0
Test examples: 1000

Example English: We humans will inevitably end up controlling our own evolution, and, because our power is emergent from nature we will make use of this acquired capacity sooner or later, for better or worse.
Reference Spanish: Nosotros humanos terminaremos inevitablemente controlando nuestra propia evolución y, puesto que nuestro poder emerge de la naturaleza, tarde o temprano utilizaremos esa capacidad adquirida, para bien o para mal.


In [19]:
# STEP 22 — Gemma temperature and sampling experiment

import os
import json
import time
import torch
import pandas as pd
import sacrebleu
from tqdm.auto import tqdm

SAMPLE_N = 100
SAMPLE_MAX_NEW_TOKENS = 96
SAMPLE_BATCH_SIZE = 4

sampling_df = test_df.iloc[:SAMPLE_N].copy().reset_index(drop=True)
sampling_prompts = [
    make_prompt(str(en))
    for en in sampling_df["en"].tolist()
]
sampling_references = sampling_df["es"].astype(str).tolist()

def generate_gemma_samples(
    prompts,
    do_sample,
    temperature=None,
    batch_size=4,
):
    generated_texts = []
    start_time = time.time()

    model.eval()

    for start in tqdm(
        range(0, len(prompts), batch_size),
        desc=f"Sampling: {temperature if do_sample else 'greedy'}",
    ):
        batch_prompts = prompts[start:start + batch_size]

        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        ).to(model.device)

        prompt_width = inputs["input_ids"].shape[1]

        generation_args = {
            "max_new_tokens": SAMPLE_MAX_NEW_TOKENS,
            "do_sample": do_sample,
            "num_beams": 1,
            "pad_token_id": tokenizer.pad_token_id,
            "eos_token_id": tokenizer.eos_token_id,
            "use_cache": True,
        }

        if do_sample:
            generation_args["temperature"] = temperature
            generation_args["top_p"] = 0.9

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                **generation_args,
            )

        new_tokens = output_ids[:, prompt_width:]
        decoded = tokenizer.batch_decode(
            new_tokens,
            skip_special_tokens=True,
        )
        generated_texts.extend([x.strip() for x in decoded])

        del inputs, output_ids, new_tokens

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.time() - start_time
    return generated_texts, elapsed

# Run greedy, temperature 0.7, and temperature 1.0
greedy_outputs, greedy_seconds = generate_gemma_samples(
    sampling_prompts,
    do_sample=False,
    batch_size=SAMPLE_BATCH_SIZE,
)

temp07_outputs, temp07_seconds = generate_gemma_samples(
    sampling_prompts,
    do_sample=True,
    temperature=0.7,
    batch_size=SAMPLE_BATCH_SIZE,
)

temp10_outputs, temp10_seconds = generate_gemma_samples(
    sampling_prompts,
    do_sample=True,
    temperature=1.0,
    batch_size=SAMPLE_BATCH_SIZE,
)

# Evaluate each setting on the same 100 examples
sampling_results = []

for name, outputs, elapsed, temperature in [
    ("greedy", greedy_outputs, greedy_seconds, None),
    ("sampling_t0.7_top_p0.9", temp07_outputs, temp07_seconds, 0.7),
    ("sampling_t1.0_top_p0.9", temp10_outputs, temp10_seconds, 1.0),
]:
    bleu_score = sacrebleu.corpus_bleu(
        outputs, [sampling_references]
    ).score

    chrf_score = sacrebleu.corpus_chrf(
        outputs, [sampling_references]
    ).score

    sampling_results.append({
        "setting": name,
        "temperature": temperature,
        "top_p": 0.9 if temperature is not None else None,
        "test_examples": SAMPLE_N,
        "BLEU": round(float(bleu_score), 4),
        "chrF": round(float(chrf_score), 4),
        "inference_seconds": round(float(elapsed), 2),
        "seconds_per_example": round(float(elapsed / SAMPLE_N), 4),
    })

sampling_results_df = pd.DataFrame(sampling_results)

sampling_output_path = os.path.join(
    GEMMA_RESULTS_DIR, "gemma_sampling_comparison.csv"
)
sampling_results_df.to_csv(sampling_output_path, index=False)

sampling_predictions_path = os.path.join(
    GEMMA_RESULTS_DIR, "gemma_sampling_predictions.csv"
)

pd.DataFrame({
    "en": sampling_df["en"],
    "reference_es": sampling_references,
    "greedy": greedy_outputs,
    "temperature_0.7_top_p_0.9": temp07_outputs,
    "temperature_1.0_top_p_0.9": temp10_outputs,
}).to_csv(sampling_predictions_path, index=False)

print("\nSAMPLING COMPARISON")
display(sampling_results_df)

print("\nSaved metrics:", sampling_output_path)
print("Saved predictions:", sampling_predictions_path)

print("\nFirst example across decoding settings:")
print("EN:", sampling_df.iloc[0]["en"])
print("Reference:", sampling_references[0])
print("Greedy:", greedy_outputs[0])
print("Temperature 0.7:", temp07_outputs[0])
print("Temperature 1.0:", temp10_outputs[0])

Sampling: greedy:   0%|          | 0/25 [00:00<?, ?it/s]

Sampling: 0.7:   0%|          | 0/25 [00:00<?, ?it/s]

Sampling: 1.0:   0%|          | 0/25 [00:00<?, ?it/s]


SAMPLING COMPARISON


,setting,temperature,top_p,test_examples,BLEU,chrF,inference_seconds,seconds_per_example
0,greedy,NaN,NaN,100,29.4521,53.3279,66.44,0.6644
1,sampling_t0.7_top_p0.9,0.7,0.9,100,24.7170,48.5303,65.66,0.6566
2,sampling_t1.0_top_p0.9,1.0,0.9,100,21.6606,45.8957,64.09,0.6409



Saved metrics: /kaggle/working/ICT4403_MT_Project/results/gemma/gemma_sampling_comparison.csv
Saved predictions: /kaggle/working/ICT4403_MT_Project/results/gemma/gemma_sampling_predictions.csv

First example across decoding settings:
EN: We humans will inevitably end up controlling our own evolution, and, because our power is emergent from nature we will make use of this acquired capacity sooner or later, for better or worse.
Reference: Nosotros humanos terminaremos inevitablemente controlando nuestra propia evolución y, puesto que nuestro poder emerge de la naturaleza, tarde o temprano utilizaremos esa capacidad adquirida, para bien o para mal.
Greedy: Nosotros, en fin, terminaremos con nosotros mismos controlando nuestra propia evolución, y, porque nuestra capacidad es emergente de la naturaleza, utilizaremos esta capacidad antes o después, para bien o mal.
Temperature 0.7: Nosotros, en última instancia, hemos de acabar controlando nuestra propia evolución, y porque nuestra capacida

In [21]:
# STEP 23 — Bundle all experiment results for final comparison

import os
import shutil

WORK_DIR = "/kaggle/working/ICT4403_MT_Project"
RESULTS_DIR = os.path.join(WORK_DIR, "results")

# Include the main model comparison CSV
comparison_csv = os.path.join(
    RESULTS_DIR, "final_model_comparison.csv"
)
assert os.path.exists(comparison_csv), (
    "Final model comparison CSV was not found."
)

# Create a bundle containing all results and metrics
bundle_path = "/kaggle/working/final_comparison_results.zip"

if os.path.exists(bundle_path):
    os.remove(bundle_path)

shutil.make_archive(
    "/kaggle/working/final_comparison_results",
    "zip",
    root_dir=WORK_DIR,
    base_dir="results",
)

print("Results bundle created:", bundle_path)
print(
    "Bundle size (MB):",
    round(os.path.getsize(bundle_path) / 1024**2, 2),
)

print("\nResult files included:")
for root, dirs, files in os.walk(RESULTS_DIR):
    for filename in files:
        print(os.path.relpath(os.path.join(root, filename), WORK_DIR))

Results bundle created: /kaggle/working/final_comparison_results.zip
Bundle size (MB): 0.1

Result files included:
results/final_model_comparison.csv
results/gemma/gemma_inference_summary.json
results/gemma/gemma_sampling_predictions.csv
results/gemma/gemma_metrics.json
results/gemma/gemma_config.json
results/gemma/gemma_sampling_comparison.csv
results/gemma/gemma_training_summary.json
results/gemma/gemma_predictions.csv


In [22]:
import os

WORK_DIR = "/kaggle/working/ICT4403_MT_Project"

for subdir in ["results", "models", "data2"]:
    path = os.path.join(WORK_DIR, subdir)
    print(f"\n{'=' * 15} {subdir.upper()} {'=' * 15}")

    if os.path.exists(path):
        for root, dirs, files in os.walk(path):
            level = root.replace(path, "").count(os.sep)
            indent = "  " * level
            print(f"{indent}{os.path.basename(root)}/")

            for filename in files:
                filepath = os.path.join(root, filename)
                size_kb = os.path.getsize(filepath) / 1024
                print(f"{indent}  {filename} ({size_kb:.1f} KB)")
    else:
        print("Directory not found.")


=============== RESULTS ===============
results/
  final_model_comparison.csv (0.4 KB)
  gemma/
    gemma_inference_summary.json (0.2 KB)
    gemma_sampling_predictions.csv (36.9 KB)
    gemma_metrics.json (0.3 KB)
    gemma_config.json (0.4 KB)
    gemma_sampling_comparison.csv (0.3 KB)
    gemma_training_summary.json (0.6 KB)
    gemma_predictions.csv (229.0 KB)
    training_logs/

=============== MODELS ===============
models/
  gemma_lora/
    final_adapter/
      adapter_model.safetensors (38336.4 KB)
      tokenizer_config.json (0.5 KB)
      adapter_config.json (1.1 KB)
      tokenizer.json (33544.5 KB)
      chat_template.jinja (0.6 KB)
      README.md (5.1 KB)
    checkpoints/
      checkpoint-625/
        scheduler.pt (1.4 KB)
        rng_state.pth (14.3 KB)
        adapter_model.safetensors (38336.4 KB)
        tokenizer_config.json (0.5 KB)
        adapter_config.json (1.1 KB)
        tokenizer.json (33544.5 KB)
        chat_template.jinja (0.6 KB)
        training_args.bi